# importación de librerías

In [1]:
import pandas as pd
import sqlite3

# Ingesta de datos

In [2]:
archivo_excel = pd.ExcelFile("data\INSUMOS OPERACIONALES STOCK RELIX.xlsx")
print("Hojas disponibles:", archivo_excel.sheet_names)

Hojas disponibles: ['Insumos Operacionales 2025', 'INGRESO', 'SALIDA', 'STOCK ']


In [3]:
stock = pd.read_excel("data\INSUMOS OPERACIONALES STOCK RELIX.xlsx", sheet_name='STOCK ', skiprows=10)
stock.head()

,Unnamed: 0,ITEM,CODIGO SAP/OTRO,DESCRIPCION INSUMO,CLASIFICACIÓN,UNIDAD,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,1.1,11174821,DUCTO RECTO;PE100;PN20;280 MM;12 M;PL,PIPING,Unidades,330.0,151.0,4.0,477.0,50.0,NaN,NaN,NaN
1,NaN,1.2,11177716,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam",PIPING,Unidades,406.0,0.0,0.0,406.0,50.0,NaN,NaN,NaN
2,NaN,1.3,en catalogacion,"Tuberia HDPE 200MM DN, Flanges Moviles C150, c...",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0,NaN,NaN,NaN
3,NaN,1.4,11151921,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam ...",PIPING,Unidades,26.0,0.0,0.0,26.0,50.0,"19 Cosapi , 7 patio 2",NaN,NaN
4,NaN,1.5,11019704,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0,NaN,NaN,NaN


### Eliminación columnas innecesarias

In [4]:
stock = stock.drop(stock.columns[[0, 11, 12, 13]], axis=1)

In [5]:
stock.head()

,ITEM,CODIGO SAP/OTRO,DESCRIPCION INSUMO,CLASIFICACIÓN,UNIDAD,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
0,1.1,11174821,DUCTO RECTO;PE100;PN20;280 MM;12 M;PL,PIPING,Unidades,330.0,151.0,4.0,477.0,50.0
1,1.2,11177716,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam",PIPING,Unidades,406.0,0.0,0.0,406.0,50.0
2,1.3,en catalogacion,"Tuberia HDPE 200MM DN, Flanges Moviles C150, c...",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0
3,1.4,11151921,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam ...",PIPING,Unidades,26.0,0.0,0.0,26.0,50.0
4,1.5,11019704,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,Unidades,NaN,0.0,0.0,0.0,50.0


In [6]:
stock.columns

Index(['ITEM', 'CODIGO SAP/OTRO', 'DESCRIPCION INSUMO', 'CLASIFICACIÓN ',
       'UNIDAD', 'STOCK INICIAL', 'INGRESOS', 'SALIDAS', 'STOCK ACTUAL',
       'PUNTO DE REORDENAMIENTO'],
      dtype='object')

In [7]:
stock.columns = stock.columns.str.strip()
stock.columns

Index(['ITEM', 'CODIGO SAP/OTRO', 'DESCRIPCION INSUMO', 'CLASIFICACIÓN',
       'UNIDAD', 'STOCK INICIAL', 'INGRESOS', 'SALIDAS', 'STOCK ACTUAL',
       'PUNTO DE REORDENAMIENTO'],
      dtype='object')

### Verificación valores nulos

In [8]:
stock.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 490 entries, 0 to 489
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ITEM                     490 non-null    float64
 1   CODIGO SAP/OTRO          490 non-null    object 
 2   DESCRIPCION INSUMO       490 non-null    object 
 3   CLASIFICACIÓN            489 non-null    object 
 4   UNIDAD                   486 non-null    object 
 5   STOCK INICIAL            185 non-null    float64
 6   INGRESOS                 489 non-null    float64
 7   SALIDAS                  486 non-null    float64
 8   STOCK ACTUAL             489 non-null    float64
 9   PUNTO DE REORDENAMIENTO  481 non-null    float64
dtypes: float64(6), object(4)
memory usage: 38.4+ KB


In [9]:
stock[stock['CLASIFICACIÓN'].isnull()]
# TODO: Llenar clasificación faltante en stock

,ITEM,CODIGO SAP/OTRO,DESCRIPCION INSUMO,CLASIFICACIÓN,UNIDAD,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
481,11.29,11220592,MAURO TUERCA STORZ 4 PULGADAS LINEA DE RIEGO,NaN,unidades,0.0,50.0,0.0,50.0,0.0


In [10]:
stock[stock['UNIDAD'].isnull()]
# TODO: llenar unidad faltante y stock??
# TODO: qué es el punto de reordenamiento?

,ITEM,CODIGO SAP/OTRO,DESCRIPCION INSUMO,CLASIFICACIÓN,UNIDAD,STOCK INICIAL,INGRESOS,SALIDAS,STOCK ACTUAL,PUNTO DE REORDENAMIENTO
166,4.40,catalogar sap,"Flange Ciego diam 28"" ANSI C150",PIEZAS ESPECIALES,NaN,NaN,0.0,10.0,-10.0,2.0
167,4.50,catalogar sap,"Flange Ciego 28"" ANSI C300",PIEZAS ESPECIALES,NaN,NaN,0.0,10.0,-10.0,2.0
321,9.68,44002471,"POLYSEAL POLIMERO;1,05G/C3;TINETA",OBRAS CIVILES,NaN,NaN,0.0,0.0,0.0,30.0
352,9.99,11215216,BOQUILLAS DEPOSITACION 50MM POLIURETANO,SERVICIOS GENERALES,NaN,NaN,0.0,0.0,0.0,50.0


In [11]:
stock['CLASIFICACIÓN'].unique()


array(['PIPING ', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACIÓN ',
       'PIEZÓMETROS ', 'IMPERMEABILIAZACIÓN ', 'OBRAS CIVILES ',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA',
       'IMPERMEABILIZACIÓN LLAU-LLAU', 'TALLER BERLIAM', nan,
       'RIEGO MURO TRANQUE', 'INSTRUMENTACIÓN GEOTECNICA'], dtype=object)